In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
  "answer": "The agentic loop keeps calling the model until it stops by wrapping the process of sending messages and running tools in a `while True` loop. The loop's core mechanism is as follows:\n\n1.  **Initialize**: The loop starts with an iteration counter and a flag, `has_function_calls`, set to `False`.\n2.  **Call the Model**: It sends the current message history (which includes instructions and the user's question, and previous model outputs/tool results) to the LLM.\n3.  **Process Response**: The model's response is received and appended to the message history. The loop then iterates through the items in the response:\n    *   **Function Call**: If an item is a `function_call`, the agent executes the corresponding tool (e.g., `search`) using the arguments provided by the model. The output of the tool call is then appended back to the message history, and the `has_function_calls` flag is set to `True`.\n    *   **Message**: If an item is a `message`, it means the model is pro

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xfd45f12f23e500b4c237b906470981da",
        "span_id": "0x0277a0ef559452b6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T14:07:39.640289Z",
    "end_time": "2026-07-20T14:07:39.640335Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "5f0af571-1f9a-40d0-aa4b-533a1e7fae24",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [3]:
from rag_helper import RAGBase
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

class RAGTraced(RAGBase):

    provider = TracerProvider()
    provider.add_span_processor(
        #SimpleSpanProcessor(ConsoleSpanExporter())
        SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
    )
    trace.set_tracer_provider(provider)

    tracer = trace.get_tracer("llm-zoomcamp")

    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage_metadata
            span.set_attribute("input_tokens", usage.prompt_token_count)
            span.set_attribute("output_tokens", usage.candidates_token_count)
            return response

    def rag(self, query):
        with self.tracer.start_as_current_span("rag"):
            return super().rag(query)

In [4]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

In [5]:
from google import genai
client = genai.Client()
rag_traced = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
  "answer": "The agentic loop keeps calling the model until it returns a response without any function calls. Inside the loop, the model's response is processed. If the response contains a `function_call` item, the corresponding tool (e.g., `search`) is executed, and its output is appended to the message history. A `has_function_calls` flag is set to `True` indicating that more calls are needed. This expanded message history, including the tool's output, is then sent back to the model in the next iteration. The loop continues this process of sending messages, executing function calls, and appending results until the model's response no longer contains any `function_call` items, at which point the `has_function_calls` flag remains `False` and the loop breaks."
}


In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [11]:
import pandas as pd

conn = sqlite3.connect("traces.db")
query= "select * from spans"
df = pd.read_sql_query(query, conn)

In [12]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784559398274354000,1784559398276121000,NaN,NaN,None
1,llm,1784559398277081000,1784559401504078000,7933.0,175.0,None
2,rag,1784559398274267000,1784559401506559000,NaN,NaN,None
3,search,1784560265326078000,1784560265337541000,NaN,NaN,None
4,llm,1784560265338553000,1784560268430311000,7297.0,7.0,None
5,rag,1784560265325988000,1784560268432481000,NaN,NaN,None
6,search,1784560268433898000,1784560268436768000,NaN,NaN,None
7,llm,1784560268438095000,1784560271673388000,7297.0,7.0,None
8,rag,1784560268433829000,1784560271675782000,NaN,NaN,None
9,search,1784560271677504000,1784560271680649000,NaN,NaN,None


In [9]:
total_duration_search = df[df['name'] == 'search']['end_time'] - df[df['name'] == 'search']['start_time']
total_duration_llm = df[df['name'] == 'llm']['end_time'] - df[df['name'] == 'llm']['start_time']
total_duration_rag = df[df['name'] == 'rag']['end_time'] - df[df['name'] == 'rag']['start_time']
print("Total duration for search:", total_duration_search.sum())
print("Total duration for llm:", total_duration_llm.sum())
print("Total duration for rag:", total_duration_rag.sum())

Total duration for search: 1767000
Total duration for llm: 3226997000
Total duration for rag: 3232292000


In [10]:
answer = rag_traced.rag(query)
print(answer)

answer = rag_traced.rag(query)
print(answer)

answer = rag_traced.rag(query)
print(answer)

"I don't know."
"I don't know."
"I don't know."
